# HarrisWGAN demo

In [1]:
import os
import sys
from typing import List, Tuple, Union
from abc import ABC, abstractmethod
from datetime import datetime as dt
from pathlib import Path
import importlib
import gc
import json as js

import numpy as np
import xarray as xr
import dask
import tensorflow as tf

repo_dir = Path("/p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/")
sys.path.append(str(repo_dir.joinpath("models")))
sys.path.append(str(repo_dir.joinpath("utils")))
sys.path.append(str(repo_dir.joinpath("handle_data")))
from model_engine import ModelEngine
#from all_normalizations import ZScore

# from handle_data_unet import HandleUnetData

### Data pipeline for input streams of Harris WGAN 

The input differs from the other baseline models in a way that the low-res input data is not upscaled (bi-linearly interpolated) to the target grid and that high-res static data constitues another input stream. Thus, the generator yields a dictionary which is then used to set-upthe TF data pipeline.
To-Do: 
 - [ ] implement updated make_tf_dataset_allmem-method in handle_data_class.py
 - [ ] adapt make_tf_dataset_dyn-method accordingly

In [ ]:
def split_in_tar(
    ds: xr.Dataset,
    predictands: List = None,
    predictors: List = None,
    static_vars: List = None,
) -> Tuple[xr.Dataset, xr.Dataset]:
    """
    Split data array with variables-dimension into input and target data for downscaling
    :param da: The unsplitted data array
    :param target_var: Name of target variable which should consttute the first channel
    :param predictands: List of selected predictand variables; parse None to use
                        all predictands (vars with suffix _tar)
    :param predictors: List of selected predictor variables; parse None to use all predictors (vars with suffix _in)
    :return: The split data array.
    """
    varnames = list(ds.data_vars)

    if predictors is None:
        invars = [var for var in varnames if var.endswith("_in")]
    else:
        assert all(
            [predictor in varnames for predictor in predictors]
        ), f"At least one predictor is not a data variable. Available variables are {*varnames,}"
        invars = list(predictors)
    if predictands is None:
        tarvars = [var for var in varnames if var.endswith("_tar")]
    else:
        assert all(
            [predictand in varnames for predictand in predictands]
        ), f"At least one predictor is not a data variable. Available variables are {*varnames,}"
        tarvars = list(predictands)

    if static_vars is None:
        ds_in, ds_tar = ds[invars], ds[tarvars]

        return ds_in, ds_tar
    else:
        assert all(
            [static_var in varnames for static_var in static_vars]
        ), f"At least ostatic high-res is not a data variable. Available variables are {*varnames,}"
        statvars = list(static_vars)

        ds_in, ds_tar, ds_stat = ds[invars], ds[tarvars], ds[statvars]

        return ds_in, ds_tar, ds_stat


def reshape_ds(ds):
    """
    Convert a xarray dataset to a data-array where the variables will constitute the last dimension (channel last)
    :param ds: the xarray dataset with dimensions (dims)
    :return da: the data-array with dimensions (dims, variables)
    """
    da = ds.to_array(dim="variables")
    da = da.transpose(..., "variables")
    return da


def make_tf_dataset_allmem(
    ds: xr.Dataset,
    batch_size: int,
    predictands: List,
    predictors: List,
    static_vars: List,
    lshuffle: bool = True,
    shuffle_samples: int = 20000,
    named_targets: bool = False,
    var_tar2in: str = None,
    lrepeat: bool = True,
    drop_remainder: bool = True,
) -> tf.data.Dataset:
    """
    Build-up TensorFlow dataset from a generator based on the xarray-data array.
    NOTE: All data is loaded into memory
    :param ds: the xarray dataset. Input variable names must carry the suffix '_in', whereas it must be '_tar' for target variables
    :param batch_size: number of samples per mini-batch
    :param predictands: List of selected predictand variables
    :param predictors: List of selected predictor variables; parse None to use all predictors (vars with suffix _in)
    :param lshuffle: flag if shuffling should be applied to dataset
    :param shuffle_samples: number of samples to load before applying shuffling
    :param named_targets: flag if target of TF dataset should be dictionary with named target variables
    :param var_tar2in: name of target variable to be added to input (used e.g. for adding high-resolved topography
                                                                        to the input)
    :param lrepeat: flag if dataset should be repeated
    :param drop_remainder: flag if samples will be dropped in case batch size is not a divisor of # data samples
    :param with_horovod: flag to trigger horovod-based distributed dataset creation
    :param lembed: flag to trigger temporal embedding (not implemented yet!)
    """

    # add time dimension to constant variables
    for var in ds.data_vars:
        if "time" not in ds[var].dims:
            ds[var] = ds[var].expand_dims({"time": ds["time"]}, axis=0)

    ds_in, ds_tar, ds_stat = split_in_tar(
        ds, predictands=predictands, predictors=predictors, static_vars=static_vars
    )

    # convert dataset to data arrays and load into memory
    da_in, da_tar, da_stat = (
        reshape_ds(ds_in).astype("float32", copy=True),
        reshape_ds(ds_tar).astype("float32", copy=True),
        reshape_ds(ds_stat).astype("float32", copy=True),
    )

    if var_tar2in is not None:
        # NOTE: * The order of the following operation must be the same as in StreamMonthlyNetCDF.getitems
        #       * The following operation order must concatenate var_tar2in by da_in to ensure
        #         that the variable appears at first place. This is required to avoid
        #         that var_tar2in becomes a predeictand when slicing takes place in tf_split
        da_in = xr.concat([da_tar.sel({"variables": var_tar2in}), da_in], "variables")

    varnames_tar = da_tar["variables"].values

    def gen_named(darr_in, darr_tar):
        # darr_in, darr_tar = darr_in.load(), darr_tar.load()
        ntimes = len(darr_in["time"])
        for t in range(ntimes):
            tar_now = darr_tar.isel({"time": t})
            yield tuple(
                (
                    darr_in.isel({"time": t}).values,
                    {
                        var: tar_now.sel({"variables": var}).values
                        for var in varnames_tar
                    },
                )
            )

    def gen_unnamed(darr_in, darr_tar):
        # darr_in, darr_tar = darr_in.load(), darr_tar.load()
        ntimes = len(darr_in["time"])
        for t in range(ntimes):
            yield tuple(
                (darr_in.isel({"time": t}).values, darr_tar.isel({"time": t}).values)
            )

    def gen_dict(darr_in, darr_tar, darr_stat):
        ntimes = len(darr_in["time"])
        for t in range(ntimes):
            yield tuple(
                (
                    {
                        "lo_res_inputs": darr_in.isel({"time": t}).values,
                        "hi_res_inputs": darr_stat.isel({"time": t}).values,
                    },
                    {"output": darr_tar.isel({"time": t}).values},
                )
            )

    if named_targets is True:
        gen_now = gen_named
    elif static_vars is not None:
        gen_now = gen_dict
    else:
        gen_now = gen_unnamed

    # create output signatures from first sample
    if static_vars is None:
        s0 = next(iter(gen_now(da_in, da_tar)))
        sample_spec_in = tf.TensorSpec(s0[0].shape, dtype=s0[0].dtype)
        if named_targets is True:
            sample_spec_tar = {
                var: tf.TensorSpec(s0[1][var].shape, dtype=s0[1][var].dtype)
                for var in varnames_tar
            }
        else:
            sample_spec_tar = tf.TensorSpec(s0[1].shape, dtype=s0[1].dtype)

        # re-instantiate the generator and build TF dataset
        gen_train = gen_now(da_in, da_tar)

    else:
        s0 = next(iter(gen_now(da_in, da_tar, da_stat)))

        sample_spec_in = {
            "lo_res_inputs": tf.TensorSpec(
                s0[0]["lo_res_inputs"].shape, dtype=s0[0]["lo_res_inputs"].dtype
            ),
            "hi_res_inputs": tf.TensorSpec(
                s0[0]["hi_res_inputs"].shape, dtype=s0[0]["hi_res_inputs"].dtype
            ),
        }

        sample_spec_tar = {
            "output": tf.TensorSpec(s0[1]["output"].shape, dtype=s0[1]["output"].dtype)
        }

        # re-instantiate the generator and build TF dataset
        gen_train = gen_now(da_in, da_tar, da_stat)

    data_iter = tf.data.Dataset.from_generator(
        lambda: gen_train, output_signature=(sample_spec_in, sample_spec_tar)
    )

    # Notes:
    # * cache is reuqired to make repeat work properly on datasets based on generators
    #   (see https://stackoverflow.com/questions/60226022/tf-data-generator-keras-repeat-does-not-work-why)
    # * repeat must be applied after shuffle to get varying mini-batches per epoch
    # * batch-size is increased to allow substepping in train_step
    if lshuffle > 1:
        data_iter = (
            data_iter.cache()
            .shuffle(shuffle_samples)
            .batch(batch_size, drop_remainder=drop_remainder)
        )
    else:
        data_iter = data_iter.cache().batch(batch_size, drop_remainder=drop_remainder)

    if lrepeat:
        data_iter = data_iter.repeat()

    # clean-up to free some memory
    # free_mem([da, da_in, da_tar, varnames_tar])
    del ds
    del ds_in
    del ds_tar
    del da_in
    del da_tar
    gc.collect()

    return data_iter

# Modified normalization class
Note that this is mandatory since the data has differing coordinates for target and input data is not yet supported by the normalization class.
To-Do:
- [ ] Revise Normalize-class accordingly
Here, an ad-hoc fix is made to allow passing of `norm_dims=None` which results into averaging over all data dimensions.

In [ ]:
da_or_ds = Union[xr.DataArray, xr.Dataset]

class Normalize(ABC):
    """
    Abstract class for normalizing data.
    """

    def __init__(self, method: str, norm_dims: List):
        self.method = method
        self.norm_dims = norm_dims
        self.norm_stats = None

    def normalize(self, data: xr.DataArray, **stats):
        """
        Normalize data
        :param data: The DataArray to be normalized
        :param stats: Optional parameters to perform normalization. Must fit to normalization type!
        :return: DataArray with normalized data
        """
        # sanity checks
        # if not isinstance(data, xr.DataArray):
        #    raise TypeError(f"Passed data must be a xarray.DataArray, but is of type {str(type(data))}.")

        # do the computation
        norm_stats = self.get_required_stats(data, **stats)
        norm_stats = Normalize.match_datatype(data, *norm_stats)
        data_norm = self.normalize_data(data, *norm_stats)

        return data_norm

    def denormalize(self, data: da_or_ds, **stats):
        """
        Denormalize data.
        :param data: The DataArray to be denormalized.
        :param stats: Optional parameters to perform denormalization. Must fit to normalization type!
        :return: DataArray with denormalized data.
        """
        # sanity checks
        # if not isinstance(data, xr.DataArray):
        #    raise TypeError(f"Passed data must be a xarray.DataArray, but is of type {str(type(data))}.")

        # do the computation
        norm_stats = self.get_required_stats(data, **stats)
        norm_stats = Normalize.match_datatype(data, *norm_stats)
        data_denorm = self.denormalize_data(data, *norm_stats)

        return data_denorm

    @property
    def norm_dims(self):
        return self._norm_dims

    @norm_dims.setter
    def norm_dims(self, norm_dims):
        self._norm_dims = list(norm_dims) if norm_dims is not None else None

    def _check_norm_dims(self, data):
        """
        Check if dimension for normalization reside in dimensions of data.
        :param data: the data (xr.DataArray) to be normalized
        :return True: in case of passed check, a ValueError is risen else
        """
        data_dims = list(data.dims)
        norm_dims_check = [norm_dim in data_dims for norm_dim in self.norm_dims]
        if not all(norm_dims_check):
            imiss = np.where(~np.array(norm_dims_check))[0]
            miss_dims = list(np.array(self.norm_dims)[imiss])
            raise ValueError("The following dimensions do not reside in the data: " +
                             f"{', '.join(miss_dims)}")

        return True

    @staticmethod
    def match_datatype(data, *args, var_dim="variables"):
        """
        Ensures that the arguments have the same xarray datatype (either xr.DataArray or xr.Dataset) as data,
        i.e. coerces all arguments against type(data) if necessary.
        :param data: the reference data (must be either xr.Dataset or xr.DataArray)
        :param args: arbitrary number of arguments (all of them must also be either xr.Dataset or xr.DataArray,
                     but should not be mixed, e.g. type(args[0])=xr.Dataset and type(args[1])=xr.DataArray is not
                     allowed
        :param var_dim: dimension name to convert from/to xr.Dataset/xr.DataArray
        """

        # sanity check
        ds_or_da = (xr.Dataset, xr.DataArray)
        all_args = [data] + list(args)
        if not all(isinstance(arg, ds_or_da) for arg in all_args):
            flags = [not isinstance(arg, ds_or_da) for arg in all_args]
            inds = np.nonzero(flags)[0].tolist()
            if len(inds) == 1:
                err_str = f"The parsed argument at position {inds} is"
            else:
                err_str = f"The parsed arguments at positions {inds} are"
            raise ValueError(f"{err_str} not an xarray.DataArray or xarray.Dataset.")

        # align type of arguments if required
        if isinstance(data, type(args[0])):
            args_new = args
        elif isinstance(data, xr.Dataset) and isinstance(args[0], xr.DataArray):
            args_new = tuple(arg.to_dataset(dim=var_dim) for arg in args)
        elif isinstance(data, xr.DataArray) and isinstance(args[0], xr.Dataset):
            args_new = tuple(arg.to_array(dim=var_dim) for arg in args)
        else:
            raise ValueError("Unknown error occured. Please check all input parameters.")

        return args_new

    def save_norm_to_file(self, js_file, missdir_ok: bool = True):
        """
        Write normalization parameters to file
        :param js_file: Path to JSON-file to be created
        :param missdir_ok: If True, base-directory of JSON-file can be missing and will be created then
        :return: -
        """
        if self.norm_stats is None:
            raise AttributeError("norm_stats is still None. Please run (de-)normalization to get parameters.")

        if any([stat is None for stat in self.norm_stats.values()]):
            raise AttributeError("Some parameters of norm_stats are None.")

        norm_serialized = {key: da.to_dict() for key, da in self.norm_stats.items()}

        # serialization and (later) deserialization depends on data type.
        # Thus, we have to save it to the dictionary
        d0 = list(self.norm_stats.values())[0]
        if isinstance(d0, xr.DataArray):
            norm_serialized["data_type"] = "data_array"
        elif isinstance(d0, xr.Dataset):
            norm_serialized["data_type"] = "data_set"

        if missdir_ok: os.makedirs(os.path.dirname(js_file), exist_ok=True)

        with open(js_file, "w") as jsf:
            js.dump(norm_serialized, jsf)

    def read_norm_from_file(self, js_file):
        """
        Read normalization parameters from file. Inverse function to write_norm_from_file.
        :param js_file: Path to JSON-file to be read.
        :return: Parameters set to self.norm_stats
        """
        with open(js_file, "r") as jsf:
            norm_data = js.load(jsf)

        data_type = norm_data.pop('data_type', None)

        if data_type == "data_array":
            xr_obj = xr.DataArray
        elif data_type == "data_set":
            xr_obj = xr.Dataset
        else:
            raise ValueError(
                f"Unknown data_type {data_type} in {js_file}. Only 'data_array' or 'data_set' are allowed.")

        norm_data.pop('data_type', None)

        norm_dict_restored = {key: xr_obj.from_dict(da_dict) for key, da_dict in norm_data.items()}

        self.norm_stats = norm_dict_restored

    @abstractmethod
    def get_required_stats(self, data, varname, *stats):
        """
        Function to retrieve either normalization parameters from data or from keyword arguments
        """
        pass

    @staticmethod
    @abstractmethod
    def normalize_data(data, *norm_param):
        """
        Function to normalize data.
        """
        pass

    @staticmethod
    @abstractmethod
    def denormalize_data(data, *norm_param):
        """
        Function to denormalize data.
        """
        pass

da_or_ds = Union[xr.DataArray, xr.Dataset]


class ZScore(Normalize):
    def __init__(self, norm_dims: List):
        super().__init__("z_score", norm_dims)
        self.norm_stats = {"mu": None, "sigma": None}

    def get_required_stats(self, data: da_or_ds, varname: str= None, **stats):
        """
        Get required parameters for z-score normalization. They are either computed from the data
        or can be parsed as keyword arguments.
        :param data: the data to be (de-)normalized
        :param varname: retrieve parameters for specific varname only (without effect if parameters must be retrieved from data)
        :param stats: keyword arguments for mean (mu) and standard deviation (std) used for normalization
        :return (mu, sigma): Parameters for normalization
        """
        mu, std = stats.get("mu", self.norm_stats["mu"]), stats.get("sigma", self.norm_stats["sigma"])

        if mu is None or std is None:
            print("Retrieve mu and sigma from data...")
            mu, std = data.mean(self.norm_dims), data.std(self.norm_dims)
            # the following ensure that both parameters are computed in one graph!
            # This significantly reduces memory footprint as we don't end up having data duplicates
            # in memory due to multiple graphs (and also seem to enfore usage of data chunks as well)
            mu, std = dask.compute(mu, std)
            self.norm_stats = {"mu": mu, "sigma": std}
        else:
            if varname:
                if isinstance(mu, xr.DataArray):
                    mu, std = mu.sel({"variables": varname}), std.sel({"variables": varname})
                elif isinstance(mu, xr.Dataset):
                    mu, std = mu[varname], std[varname]
                else:
                    raise ValueError(f"Unexpected data type for mu and std: {type(mu)}, {type(std)}")
        #    print("Mu and sigma are parsed for (de-)normalization.")

        return mu, std

    @staticmethod
    def normalize_data(data, mu, std):
        """
        Perform z-score normalization on data
        :param data: Data array of interest
        :param mu: mean of data for normalization
        :param std: standard deviation of data for normalization
        :return data_norm: normalized data
        """
        data = (data - mu) / std

        return data

    @staticmethod
    def denormalize_data(data, mu, std):
        """
        Perform z-score denormalization on data.
        :param data: Data array of interest
        :param mu: mean of data for denormalization
        :param std: standard deviation of data for denormalization
        :return data_norm: denormalized data
        """
        data = data * std + mu

        return data

### Get the data
Stream data from example data-file.

In [ ]:
# set diretcories and (hyper-)parameters for WGAN
data_dir = Path("/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/dataset/testdata/")
t2m_test_file = data_dir.joinpath("downscaling_benchmark_t2m_allmem_test.nc")
js_norm = data_dir.joinpath("norm.json")
# datadir = "/p/scratch/deepacf/maelstrom/maelstrom_data/ap5_michael/preprocessed_era5_ifs/netcdf_data/all_files/"
# outdir = "/p/project/deepacf/maelstrom/langguth1/downscaling_jsc_repo/downscaling_unet/trained_models"

lr_gen = 5.0e-05
lr_gen_end = lr_gen / 10.0
lr_critic = 1.0e-06
lr_decay = True
nepochs = 1
d_steps = 5
batch_size_demo = 2

# get normalization instance
data_norm = ZScore(None)
data_norm.read_norm_from_file(js_norm)

# read raw data...
ds_train = xr.open_dataset(t2m_test_file)

#...and normalize
ds_train = data_norm.normalize(ds_train)
ds_val = ds_train
z_branch = False
print("Datasets for trining, validation and testing loaded.")

# wgan_model = HarrisWGAN(GeneratorHarris, DiscriminatorHarris,
#                  {"lr_decay": lr_decay, "lr_gen": lr_gen, "lr_critic": lr_critic, "lr_gen_end": lr_gen_end,
#                   "train_epochs": nepochs, "d_steps": d_steps, "z_branch": z_branch})

Set-up data pipeline (same for training and validation for simplicity)

In [ ]:
tfds = make_tf_dataset_allmem(
    ds_train,
    batch_size_demo * (d_steps + 1),
    ["t_2m_tar"],
    ["t2m_in", "sp_in", "sshf_in", "t115_in"],
    ["fr_land_tar", "hsurf_tar"],
)
tfds_val = make_tf_dataset_allmem(
    ds_train,
    batch_size_demo * (d_steps + 1),
    ["t_2m_tar"],
    ["t2m_in", "sp_in", "sshf_in", "t115_in"],
    ["fr_land_tar", "hsurf_tar"],
)

In [ ]:
tfds, tfds_val

### Start training 

In [2]:
importlib.reload(sys.modules["model_engine"])
importlib.reload(sys.modules["harris_wgan_model"])

# some prerequisites to instantiate the model and run training
shape_in = {
    "harris_generator": {
        "lo_res_inputs": (32, 36, 4),
        "hi_res_inputs": (128, 144, 2),
        "noise_input": (32, 36, 4),
    },
    "harris_discriminator": {
        "lo_res_inputs": (32, 36, 4),
        "hi_res_inputs": (128, 144, 2),
        "output": (128, 144, 1),
    },
}
varnames_tar = "t_2m_tar"
hparams_dict = dict()                    # make use of defaults!
model_savedir = ""
steps_per_epoch = 100

model_instance = ModelEngine("harris_wgan")
# data prep here
model = model_instance(
    shape_in, list(varnames_tar), hparams_dict, model_savedir, "demo"
)
model.compile(**model.compile_options)
history = model.fit(
    x=tfds,
    epochs=model.hparams["nepochs"],
    steps_per_epoch=steps_per_epoch,
    validation_data=tfds_val,
    validation_steps=300,
    verbose=1,
    **model.fit_options
)

generator_input shape: (None, 32, 36, 4)
constants_input shape: (None, 128, 144, 2)


2024-07-19 10:18:56.248060: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14782 MB memory:  -> device: 0, name: Tesla V100-SXM2-16GB, pci bus id: 0000:60:00.0, compute capability: 7.0
2024-07-19 10:18:56.248864: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 14782 MB memory:  -> device: 1, name: Tesla V100-SXM2-16GB, pci bus id: 0000:61:00.0, compute capability: 7.0
2024-07-19 10:18:56.249424: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 14782 MB memory:  -> device: 2, name: Tesla V100-SXM2-16GB, pci bus id: 0000:88:00.0, compute capability: 7.0
2024-07-19 10:18:56.249969: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 14782 MB memory:  -> device: 3, name: Tesla V100-SXM2-16GB, pci bus id

upscaled constants shape: (None, 32, 36, 128)
noise_input shape: (None, 32, 36, 4)
Shape after first concatenate: (None, 32, 36, 136)
End of first residual block
Shape after first residual block: (None, 32, 36, 128)
Shape after upsampling step 1: (None, 128, 144, 128)
Shape after residual block: (None, 128, 144, 128)
Shape after second concatenate: (None, 128, 144, 130)
Shape after third residual block: (None, 128, 144, 128)
Output shape: (None, 128, 144, 1)
generator_input shape: (None, 32, 36, 4)
constants_input shape: (None, 128, 144, 2)
generator_output shape: (None, 128, 144, 1)
upscaled constants shape: (None, 32, 36, 512)
Shape after lo-res concatenate: (None, 32, 36, 516)
Shape after hi-res concatenate: (None, 128, 144, 3)
Shape of lo-res input after residual block: (None, 32, 36, 1024)
Shape of hi_res_input after upsampling step 1: (None, 32, 36, 1024)
Shape of hi-res input after residual block: (None, 32, 36, 1024)
Shape after concatenating lo-res input and hi-res input: (Non

NameError: name 'tfds' is not defined

In [ ]:
model_name = "harriswgan_lr1e-05_epochs1_opt_split_era5_ifs"

savedir = os.path.join("../downscaling_harriswgan/trained_models/", model_name)
os.makedirs(savedir, exist_ok=True)

In [ ]:
model.generator.save(
    os.path.join(savedir, "harriswgan_lr1e-05_epochs30_demo_generator")
)
model.critic.save(
    os.path.join(savedir, "harriswgan_lr1e-05_epochs30_demo_discriminator")
)